# Pythonic Thinking & Design Patterns

Python에서 자주 쓰는 코드 관례를 예제로 살펴본다. Decorator로 공통 기능을 묶고, Singleton·Factory·Observer·Strategy 패턴을 구현한다. Type Hint와 Docstring으로 코드의 사용 방법을 남긴다.


# 1. Pythonic Idioms (50분)

Pythonic은 Python의 문법과 관례를 활용해 의도를 읽기 쉽게 표현하는 코드 스타일을 뜻한다. 무조건 짧게 쓰는 것보다 함께 읽는 사람이 동작을 이해할 수 있는지가 기준이다.

The Zen of Python에서는 단순함, 명시성, 가독성 등을 강조한다. 이번 절에서는 같은 작업을 여러 방식으로 작성하고 그 차이를 비교한다.


In [ ]:
# The Zen of Python
import this

## 1.1 EAFP vs LBYL

LBYL(Look Before You Leap)은 조건을 먼저 확인한 뒤 실행한다. EAFP(Easier to Ask for Forgiveness than Permission)는 우선 실행하고 예상한 예외를 처리한다.

아래 예제에서 검사와 실행이 중복되는 부분, 정상 흐름과 오류 처리의 구분을 비교해 보자. 확인과 실행 사이의 상태 변화도 고려해야 한다. 어느 방식이 빠른지는 실패 빈도와 작업에 따라 달라진다.


In [ ]:
# LBYL vs EAFP 비교
print("=== LBYL vs EAFP ===")

data = {'name': 'Alice', 'age': 25}

# LBYL - Not Pythonic
print("\n1. LBYL (Look Before You Leap):")
if 'email' in data:
    email = data['email']
    print(f"Email: {email}")
else:
    email = 'N/A'
    print(f"Email not found, using default: {email}")

# EAFP - Pythonic!
print("\n2. EAFP (Easier to Ask for Forgiveness than Permission):")
try:
    email = data['email']
    print(f"Email: {email}")
except KeyError:
    email = 'N/A'
    print(f"Email not found, using default: {email}")

In [ ]:
# 복잡한 케이스에서 EAFP의 우수성
print("\n=== 복잡한 케이스 비교 ===")

data = {
    'users': [
        {'name': 'Alice', 'scores': [85, 90, 92]},
        {'name': 'Bob', 'scores': []}
    ]
}

# LBYL - 복잡하고 읽기 어려움
print("1. LBYL (복잡함):")
if 'users' in data and isinstance(data['users'], list) and len(data['users']) > 0:
    user = data['users'][0]
    if 'scores' in user and isinstance(user['scores'], list) and len(user['scores']) > 0:
        first_score = user['scores'][0]
        print(f"   First score: {first_score}")
    else:
        print("   No scores available")
else:
    print("   No users available")

# EAFP - 간결하고 읽기 쉬움
print("\n2. EAFP (간결함):")
try:
    first_score = data['users'][0]['scores'][0]
    print(f"   First score: {first_score}")
except (KeyError, IndexError, TypeError):
    print("   No score available")

In [ ]:
# Pythonic한 dict 접근 방법
print("\n=== Pythonic Dict 접근 ===")

config = {'host': 'localhost', 'port': 8080}

# Bad: LBYL
if 'timeout' in config:
    timeout = config['timeout']
else:
    timeout = 30

# Good: dict.get() with default
timeout = config.get('timeout', 30)
print(f"Timeout: {timeout}")

# Good: dict.setdefault()
# 키가 없으면 기본값을 설정하고 반환
timeout = config.setdefault('timeout', 30)
print(f"Timeout (setdefault): {timeout}")
print(f"Config: {config}")

## 1.2 Duck Typing

Duck Typing은 구체적인 타입 이름보다 필요한 동작을 지원하는지를 기준으로 객체를 사용한다. 같은 메서드를 제공하면 서로 다른 클래스의 객체도 같은 코드에서 다룰 수 있다.

Python의 프로토콜도 이런 방식으로 쓰인다. 예를 들어 `__len__`은 `len()`, `__iter__`는 반복 구문과 연결된다. 아래 예제에서 타입 확인 없이 공통 동작을 호출해 보자.


In [ ]:
# Duck Typing 기본 예시
print("=== Duck Typing ===")

class Duck:
    def quack(self):
        return "Quack! Quack!"
    
    def fly(self):
        return "Flap flap! Flying away..."

class Person:
    def quack(self):
        return "I'm imitating a duck: Quack!"
    
    def fly(self):
        return "I'm flapping my arms!"

class Robot:
    def quack(self):
        return "*ROBOTIC QUACKING SOUND*"
    
    def fly(self):
        return "*ACTIVATING THRUSTERS*"

# Duck Typing: 타입을 체크하지 않고 메서드만 호출
def make_it_quack_and_fly(duck_like_thing):
    """Duck처럼 행동하는 객체면 뭐든 OK!"""
    print(f"Quacking: {duck_like_thing.quack()}")
    print(f"Flying: {duck_like_thing.fly()}")
    print()

# 모두 작동함!
duck = Duck()
person = Person()
robot = Robot()

print("1. Duck:")
make_it_quack_and_fly(duck)

print("2. Person:")
make_it_quack_and_fly(person)

print("3. Robot:")
make_it_quack_and_fly(robot)

In [ ]:
# Protocol-Oriented Programming
print("=== Protocol-Oriented Programming ===")

class MyList:
    """리스트처럼 동작하는 커스텀 클래스"""
    def __init__(self, data):
        self._data = list(data)
    
    def __len__(self):
        """len() 프로토콜 구현"""
        return len(self._data)
    
    def __getitem__(self, index):
        """인덱싱 프로토콜 구현"""
        return self._data[index]
    
    def __iter__(self):
        """반복 프로토콜 구현"""
        return iter(self._data)

my_list = MyList([1, 2, 3, 4, 5])

# len() 사용 가능 (__len__ 구현)
print(f"Length: {len(my_list)}")

# 인덱싱 사용 가능 (__getitem__ 구현)
print(f"First item: {my_list[0]}")

# for 루프 사용 가능 (__iter__ 구현)
print("Iterating:")
for item in my_list:
    print(f"  {item}")

# 슬라이싱도 가능 (__getitem__ 덕분)
print(f"Slice [1:3]: {list(my_list[1:3])}")

In [ ]:
# Bad: Type Checking (Anti-Pattern)
print("\n=== Type Checking: Bad vs Good ===")

# Bad: 타입을 명시적으로 체크
def process_bad(obj):
    if isinstance(obj, list):
        return len(obj)
    elif isinstance(obj, dict):
        return len(obj)
    elif isinstance(obj, str):
        return len(obj)
    else:
        raise TypeError("Unsupported type")

# Good: Duck Typing - 그냥 시도
def process_good(obj):
    """len()을 지원하는 객체면 뭐든 OK!"""
    try:
        return len(obj)
    except TypeError:
        raise TypeError(f"{type(obj).__name__} doesn't support len()")

# 테스트
test_objects = [
    [1, 2, 3],
    {'a': 1, 'b': 2},
    "hello",
    MyList([1, 2, 3, 4])
]

print("Good 방식으로 처리:")
for obj in test_objects:
    result = process_good(obj)
    print(f"  {type(obj).__name__}: length = {result}")

## 1.3 Comprehension과 Generator Expression

- List: `[x for x in iterable if condition]`
- Dict: `{k: v for k, v in iterable if condition}`
- Set: `{x for x in iterable if condition}`
- Generator: `(x for x in iterable if condition)`

필터링과 변환을 한눈에 읽을 수 있을 때 사용한다. 조건이나 중첩이 복잡해지면 일반 반복문으로 풀어 쓰는 편이 나을 수 있다.


In [ ]:
# Comprehensions
print("=== Comprehensions ===")

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Bad: Traditional loop
print("1. Bad - Traditional loop:")
squares = []
for n in numbers:
    if n % 2 == 0:
        squares.append(n ** 2)
print(f"   Even squares: {squares}")

# Good: List comprehension
print("\n2. Good - List comprehension:")
squares = [n ** 2 for n in numbers if n % 2 == 0]
print(f"   Even squares: {squares}")

# Dict comprehension
print("\n3. Dict comprehension:")
square_dict = {n: n ** 2 for n in numbers if n % 2 == 0}
print(f"   Square dict: {square_dict}")

# Set comprehension
print("\n4. Set comprehension:")
words = ['apple', 'banana', 'apple', 'cherry', 'banana']
unique_lengths = {len(word) for word in words}
print(f"   Unique lengths: {unique_lengths}")

# Generator expression (메모리 효율적)
print("\n5. Generator expression:")
squares_gen = (n ** 2 for n in numbers if n % 2 == 0)
print(f"   Generator: {squares_gen}")
print(f"   Values: {list(squares_gen)}")

In [ ]:
# 중첩 Comprehension
print("\n=== 중첩 Comprehension ===")

# 2D 리스트 평탄화
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

# Bad
flat = []
for row in matrix:
    for item in row:
        flat.append(item)

# Good
flat = [item for row in matrix for item in row]
print(f"Flattened: {flat}")

# 조건부 중첩 comprehension
pairs = [(x, y) for x in range(3) for y in range(3) if x != y]
print(f"Pairs (x != y): {pairs}")

In [ ]:
# Pythonic 패턴들
print("\n=== 기타 Pythonic 패턴 ===")

# 1. enumerate() 사용
print("1. enumerate() - 인덱스와 값 동시에:")
fruits = ['apple', 'banana', 'cherry']
for i, fruit in enumerate(fruits):
    print(f"   {i}: {fruit}")

# 2. zip() 사용
print("\n2. zip() - 여러 시퀀스 동시 순회:")
names = ['Alice', 'Bob', 'Charlie']
ages = [25, 30, 35]
for name, age in zip(names, ages):
    print(f"   {name}: {age}세")

# 3. unpacking
print("\n3. Unpacking:")
a, b, c = [1, 2, 3]
print(f"   a={a}, b={b}, c={c}")

first, *middle, last = [1, 2, 3, 4, 5]
print(f"   first={first}, middle={middle}, last={last}")

# 4. 삼항 연산자
print("\n4. 삼항 연산자:")
age = 20
status = "성인" if age >= 18 else "미성년자"
print(f"   나이 {age}: {status}")

# 5. 다중 비교
print("\n5. 다중 비교:")
x = 5
if 0 < x < 10:
    print(f"   {x}는 0과 10 사이")

## 1.4 Context Manager

Context Manager는 `with` 블록에 들어갈 때와 나올 때 필요한 동작을 묶는다. 클래스에서는 `__enter__`와 `__exit__`를 구현한다. 파일 닫기, 락 해제, 연결 정리, 실행 시간 측정 등에 사용한다.

호출한 쪽에서 매번 정리 코드를 작성하지 않아도 되며, 블록 안에서 예외가 발생한 경우의 처리를 한곳에 둘 수 있다.


In [ ]:
# Context Manager 직접 구현
print("=== Custom Context Manager ===")

import time

class Timer:
    """코드 실행 시간 측정 Context Manager"""
    
    def __enter__(self):
        """with 블록 진입 시 호출"""
        print("Timer started...")
        self.start = time.time()
        return self  # with ... as 변수에 할당될 객체
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """with 블록 종료 시 호출 (예외 발생 시에도 실행)"""
        self.end = time.time()
        elapsed = self.end - self.start
        print(f"Timer stopped. Elapsed: {elapsed:.4f}s")
        
        # False 반환 시 예외 전파, True 반환 시 예외 억제
        return False

# 사용 예시
with Timer():
    # 시간을 측정할 코드
    total = 0
    for i in range(1000000):
        total += i
    print(f"Sum: {total}")

In [ ]:
# contextlib로 간단하게 구현
print("\n=== contextlib 사용 ===")

from contextlib import contextmanager
import time

@contextmanager
def timer(name="Operation"):
    """함수 기반 Context Manager"""
    print(f"{name} started...")
    start = time.time()
    try:
        yield  # with 블록 실행
    finally:
        elapsed = time.time() - start
        print(f"{name} completed in {elapsed:.4f}s")

# 사용
with timer("Heavy computation"):
    result = sum(i ** 2 for i in range(100000))
    print(f"Result: {result}")

In [ ]:
# 실용적인 예시: 디렉토리 변경
print("\n=== 실용적 Context Manager ===")

import os
from contextlib import contextmanager

@contextmanager
def change_directory(path):
    """임시로 디렉토리 변경"""
    original = os.getcwd()
    try:
        os.chdir(path)
        print(f"Changed to: {os.getcwd()}")
        yield
    finally:
        os.chdir(original)
        print(f"Restored to: {os.getcwd()}")

print(f"Current directory: {os.getcwd()}")

# /tmp로 이동했다가 자동으로 복원
with change_directory('/tmp'):
    print(f"Working in: {os.getcwd()}")

print(f"Back to: {os.getcwd()}")

# 2. Decorator (50분)

## 2.1 Function Decorator 기초

Decorator는 함수를 받아 다른 함수를 반환하는 방식으로, 원래 함수에 공통 동작을 덧붙일 수 있다. `@decorator` 문법은 함수 정의에 이를 적용한다.

```python
def decorator(func):
    def wrapper(*args, **kwargs):
        # 원래 함수 실행 전의 작업
        result = func(*args, **kwargs)
        # 원래 함수 실행 후의 작업
        return result
    return wrapper
```

로깅이나 실행 시간 측정처럼 여러 함수에서 반복되는 작업을 묶을 때 사용할 수 있다.


In [ ]:
# Decorator 기본 예시
print("=== 기본 Decorator ===")

def simple_decorator(func):
    """가장 기본적인 Decorator"""
    def wrapper():
        print("함수 실행 전")
        func()
        print("함수 실행 후")
    return wrapper

# 수동으로 적용
def greet():
    print("Hello!")

decorated_greet = simple_decorator(greet)
decorated_greet()

# @ 문법 사용 (위와 동일)
print("\n@ 문법 사용:")

@simple_decorator
def goodbye():
    print("Goodbye!")

goodbye()

In [ ]:
# 실용적인 Decorator: 타이머
print("\n=== 타이머 Decorator ===")

import time
from functools import wraps

def timer(func):
    """함수 실행 시간 측정"""
    @wraps(func)  # 원본 함수의 메타데이터 보존
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_function():
    """느린 함수 시뮬레이션"""
    time.sleep(0.5)
    return "Done"

@timer
def compute_sum(n):
    """합계 계산"""
    return sum(range(n))

# 테스트
result1 = slow_function()
print(f"Result: {result1}\n")

result2 = compute_sum(1000000)
print(f"Sum: {result2}")

In [ ]:
# @wraps의 중요성
print("\n=== @wraps의 중요성 ===")

def decorator_without_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def decorator_with_wraps(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator_without_wraps
def function_a():
    """Function A의 독스트링"""
    pass

@decorator_with_wraps
def function_b():
    """Function B의 독스트링"""
    pass

print(f"Without @wraps:")
print(f"  Name: {function_a.__name__}")
print(f"  Doc: {function_a.__doc__}")

print(f"\nWith @wraps:")
print(f"  Name: {function_b.__name__}")
print(f"  Doc: {function_b.__doc__}")

## 2.2 인자를 받는 Decorator

Decorator의 동작을 설정하려면 인자를 받는 Factory를 한 겹 더 둔다. Factory는 설정값을 받고, Decorator는 함수를 받으며, Wrapper는 실제 호출을 처리한다.

```python
def decorator_factory(arg1, arg2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # 여기서 arg1, arg2를 사용할 수 있다.
            return func(*args, **kwargs)
        return wrapper
    return decorator
```


In [ ]:
# Decorator with Arguments
print("=== Decorator with Arguments ===")

def repeat(n):
    """함수를 n번 반복 실행하는 Decorator"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(n):
                print(f"Iteration {i + 1}:")
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    print(f"  Hello, {name}!")

greet("Alice")

In [ ]:
# 실용적인 예시: 로깅 레벨
print("\n=== 로깅 Decorator ===")

def log(level="INFO"):
    """로깅 레벨을 지정할 수 있는 Decorator"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[{level}] Calling {func.__name__} with args={args}, kwargs={kwargs}")
            result = func(*args, **kwargs)
            print(f"[{level}] {func.__name__} returned {result}")
            return result
        return wrapper
    return decorator

@log(level="DEBUG")
def add(a, b):
    return a + b

@log(level="ERROR")
def divide(a, b):
    return a / b

# 테스트
result1 = add(3, 5)
print()
result2 = divide(10, 2)

In [ ]:
# Retry Decorator (실전 예시)
print("\n=== Retry Decorator ===")

import time

def retry(max_attempts=3, delay=1):
    """실패 시 재시도하는 Decorator"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt + 1}/{max_attempts} failed: {e}")
                    if attempt < max_attempts - 1:
                        print(f"Retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        print("All attempts failed")
                        raise
        return wrapper
    return decorator

# 시뮬레이션: 2번 실패 후 성공
attempt_count = 0

@retry(max_attempts=3, delay=0.5)
def unreliable_function():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ValueError("Simulated failure")
    return "Success!"

result = unreliable_function()
print(f"\nFinal result: {result}")

## 2.3 Class Decorator

Class Decorator는 클래스 정의에 적용된다. 클래스를 받아 속성이나 메서드를 추가하거나, 클래스 등록 같은 작업을 한 뒤 결과를 반환한다. 아래 예제에서는 Singleton 구현과 등록 방식을 살펴본다.


In [ ]:
# Class Decorator: Singleton 구현
print("=== Singleton Class Decorator ===")

def singleton(cls):
    """Singleton 패턴을 구현하는 Class Decorator"""
    instances = {}
    
    @wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    
    return get_instance

@singleton
class Database:
    def __init__(self):
        print("Connecting to database...")
        self.connection = "Connected"

# 첫 번째 인스턴스 생성
db1 = Database()
print(f"db1: {db1}")

# 두 번째 "생성" - 실제로는 같은 인스턴스 반환
db2 = Database()
print(f"db2: {db2}")

# 같은 인스턴스인지 확인
print(f"\ndb1 is db2: {db1 is db2}")

In [ ]:
# Class Decorator: 메서드 추가
print("\n=== 메서드 추가 Decorator ===")

def add_repr(cls):
    """클래스에 __repr__ 메서드 자동 추가"""
    def __repr__(self):
        attrs = ', '.join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    
    cls.__repr__ = __repr__
    return cls

@add_repr
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

person = Person("Alice", 25)
print(person)  # 자동으로 추가된 __repr__ 사용

## 2.4 functools 모듈

### 주요 기능

- `@wraps`: 메타데이터 보존
- `@lru_cache`: 메모이제이션
- `@singledispatch`: 타입별 다른 구현
- `@total_ordering`: 비교 메서드 자동 생성


In [ ]:
# @lru_cache: 메모이제이션
print("=== @lru_cache ===")

from functools import lru_cache
import time

# 캐시 없이
def fibonacci_slow(n):
    if n < 2:
        return n
    return fibonacci_slow(n-1) + fibonacci_slow(n-2)

# 캐시 사용
@lru_cache(maxsize=128)
def fibonacci_fast(n):
    if n < 2:
        return n
    return fibonacci_fast(n-1) + fibonacci_fast(n-2)

# 성능 비교
print("Without cache:")
start = time.time()
result = fibonacci_slow(30)
print(f"  fibonacci(30) = {result}")
print(f"  Time: {time.time() - start:.4f}s")

print("\nWith @lru_cache:")
start = time.time()
result = fibonacci_fast(30)
print(f"  fibonacci(30) = {result}")
print(f"  Time: {time.time() - start:.4f}s")

# 캐시 정보 확인
print(f"\nCache info: {fibonacci_fast.cache_info()}")

In [ ]:
# @singledispatch: 타입별 다른 구현
print("\n=== @singledispatch ===")

from functools import singledispatch

@singledispatch
def process(data):
    """기본 구현"""
    print(f"Processing {type(data).__name__}: {data}")

@process.register(list)
def _(data):
    """리스트 처리"""
    print(f"Processing list with {len(data)} items")
    for item in data:
        print(f"  - {item}")

@process.register(dict)
def _(data):
    """딕셔너리 처리"""
    print(f"Processing dict with {len(data)} keys")
    for key, value in data.items():
        print(f"  {key}: {value}")

@process.register(int)
@process.register(float)
def _(data):
    """숫자 처리"""
    print(f"Processing number: {data}")
    print(f"  Squared: {data ** 2}")

# 테스트
process("hello")  # 기본 구현
print()
process([1, 2, 3])  # 리스트 구현
print()
process({'a': 1, 'b': 2})  # 딕셔너리 구현
print()
process(42)  # 숫자 구현

# 3. Design Patterns (50분)

## 3.1 Creational Patterns

### Singleton Pattern

Singleton은 인스턴스를 하나만 만들어 공유하려는 패턴이다. 설정, 로거, 캐시처럼 공유할 상태가 있을 때 검토한다. 아래 구현에서는 여러 번 생성 요청을 해도 같은 객체를 반환하는지 확인한다.


In [ ]:
# Singleton Pattern - __new__ 사용
print("=== Singleton Pattern ===")

class Singleton:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("Creating new instance")
            cls._instance = super().__new__(cls)
        else:
            print("Returning existing instance")
        return cls._instance
    
    def __init__(self, value=None):
        # 주의: __init__은 매번 호출됨
        if not hasattr(self, 'initialized'):
            self.value = value
            self.initialized = True
            print(f"Initialized with value: {value}")

# 테스트
s1 = Singleton("first")
print(f"s1.value: {s1.value}\n")

s2 = Singleton("second")
print(f"s2.value: {s2.value}")

print(f"\ns1 is s2: {s1 is s2}")

In [ ]:
# 실용적인 Singleton: Logger
print("\n=== Singleton Logger ===")

class Logger:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.logs = []
        return cls._instance
    
    def log(self, message):
        self.logs.append(message)
        print(f"[LOG] {message}")
    
    def get_logs(self):
        return self.logs

# 여러 곳에서 Logger 사용
logger1 = Logger()
logger1.log("Application started")

logger2 = Logger()
logger2.log("User logged in")

# 같은 인스턴스이므로 로그 공유
print(f"\nAll logs: {logger1.get_logs()}")
print(f"logger1 is logger2: {logger1 is logger2}")

### Factory Pattern

Factory는 어떤 객체를 만들지 결정하는 코드를 한곳에 모은다. 호출한 쪽은 구체적인 클래스의 생성 과정 대신 Factory가 제공하는 인터페이스를 사용한다. 실행 중 설정에 따라 객체 종류를 고를 때 활용할 수 있다.


In [ ]:
# Factory Pattern
print("=== Factory Pattern ===")

from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def speak(self):
        pass
    
    @abstractmethod
    def move(self):
        pass

class Dog(Animal):
    def speak(self):
        return "Woof!"
    
    def move(self):
        return "Running on four legs"

class Cat(Animal):
    def speak(self):
        return "Meow!"
    
    def move(self):
        return "Walking gracefully"

class Bird(Animal):
    def speak(self):
        return "Chirp!"
    
    def move(self):
        return "Flying in the sky"

class AnimalFactory:
    """동물 객체를 생성하는 Factory"""
    
    @staticmethod
    def create(animal_type: str) -> Animal:
        animals = {
            'dog': Dog,
            'cat': Cat,
            'bird': Bird
        }
        
        animal_class = animals.get(animal_type.lower())
        if animal_class is None:
            raise ValueError(f"Unknown animal type: {animal_type}")
        
        return animal_class()

# Factory 사용
for animal_type in ['dog', 'cat', 'bird']:
    animal = AnimalFactory.create(animal_type)
    print(f"{animal_type.capitalize()}: {animal.speak()} - {animal.move()}")

In [ ]:
# 실용적인 예시: Document Factory
print("\n=== Document Factory ===")

class Document(ABC):
    @abstractmethod
    def export(self):
        pass

class PDFDocument(Document):
    def __init__(self, content):
        self.content = content
    
    def export(self):
        return f"Exporting to PDF: {self.content}"

class WordDocument(Document):
    def __init__(self, content):
        self.content = content
    
    def export(self):
        return f"Exporting to Word: {self.content}"

class HTMLDocument(Document):
    def __init__(self, content):
        self.content = content
    
    def export(self):
        return f"Exporting to HTML: {self.content}"

class DocumentFactory:
    @staticmethod
    def create_document(doc_type: str, content: str) -> Document:
        documents = {
            'pdf': PDFDocument,
            'word': WordDocument,
            'html': HTMLDocument
        }
        
        doc_class = documents.get(doc_type.lower())
        if doc_class is None:
            raise ValueError(f"Unknown document type: {doc_type}")
        
        return doc_class(content)

# 사용
content = "Hello, World!"
for doc_type in ['pdf', 'word', 'html']:
    doc = DocumentFactory.create_document(doc_type, content)
    print(doc.export())

## 3.2 Behavioral Patterns

### Observer Pattern

Observer는 한 객체의 변경을 구독 중인 여러 객체에 알리는 패턴이다. 이벤트 처리나 데이터 변경 알림에 사용한다. 예제에서는 구독자를 등록·해제하고 변경 알림을 전달한다.


In [ ]:
# Observer Pattern
print("=== Observer Pattern ===")

class Subject:
    """관찰 대상 (Publisher)"""
    
    def __init__(self):
        self._observers = []
        self._state = None
    
    def attach(self, observer):
        """Observer 등록"""
        if observer not in self._observers:
            self._observers.append(observer)
            print(f"{observer.name} subscribed")
    
    def detach(self, observer):
        """Observer 제거"""
        if observer in self._observers:
            self._observers.remove(observer)
            print(f"{observer.name} unsubscribed")
    
    def notify(self, message):
        """모든 Observer에게 알림"""
        print(f"\nNotifying {len(self._observers)} observers...")
        for observer in self._observers:
            observer.update(message)
    
    def set_state(self, state):
        """상태 변경 및 알림"""
        self._state = state
        self.notify(f"State changed to: {state}")

class Observer:
    """관찰자 (Subscriber)"""
    
    def __init__(self, name):
        self.name = name
    
    def update(self, message):
        """알림 받았을 때 실행"""
        print(f"  [{self.name}] received: {message}")

# 사용 예시
subject = Subject()

# Observer 생성 및 등록
observer1 = Observer("Observer 1")
observer2 = Observer("Observer 2")
observer3 = Observer("Observer 3")

subject.attach(observer1)
subject.attach(observer2)
subject.attach(observer3)

# 상태 변경 → 모든 Observer에게 알림
subject.set_state("Active")

# Observer 제거
print()
subject.detach(observer2)

# 다시 상태 변경
subject.set_state("Inactive")

In [ ]:
# 실용적인 예시: 뉴스 구독
print("\n=== 뉴스 구독 시스템 ===")

class NewsPublisher:
    """뉴스 발행자"""
    
    def __init__(self, name):
        self.name = name
        self._subscribers = []
    
    def subscribe(self, subscriber):
        self._subscribers.append(subscriber)
        print(f"{subscriber.name} subscribed to {self.name}")
    
    def unsubscribe(self, subscriber):
        self._subscribers.remove(subscriber)
        print(f"{subscriber.name} unsubscribed from {self.name}")
    
    def publish(self, article):
        print(f"\n[{self.name}] Publishing: {article}")
        for subscriber in self._subscribers:
            subscriber.receive(self.name, article)

class Subscriber:
    """구독자"""
    
    def __init__(self, name):
        self.name = name
        self.articles = []
    
    def receive(self, publisher, article):
        self.articles.append((publisher, article))
        print(f"  {self.name} received: {article}")

# 사용
tech_news = NewsPublisher("Tech News")
sports_news = NewsPublisher("Sports News")

alice = Subscriber("Alice")
bob = Subscriber("Bob")

tech_news.subscribe(alice)
tech_news.subscribe(bob)
sports_news.subscribe(bob)

tech_news.publish("Python 3.12 Released!")
sports_news.publish("Team wins championship!")

### Strategy Pattern

Strategy는 바꾸어 사용할 알고리즘을 공통 인터페이스로 묶는다. 결제, 정렬, 압축처럼 같은 목적의 처리 방법을 실행 중에 선택할 때 사용한다.


In [ ]:
# Strategy Pattern
print("=== Strategy Pattern ===")

from abc import ABC, abstractmethod

class PaymentStrategy(ABC):
    """결제 전략 인터페이스"""
    
    @abstractmethod
    def pay(self, amount):
        pass

class CreditCardPayment(PaymentStrategy):
    def __init__(self, card_number):
        self.card_number = card_number
    
    def pay(self, amount):
        return f"Paid ${amount} with credit card {self.card_number[-4:]}****"

class PayPalPayment(PaymentStrategy):
    def __init__(self, email):
        self.email = email
    
    def pay(self, amount):
        return f"Paid ${amount} via PayPal ({self.email})"

class BitcoinPayment(PaymentStrategy):
    def __init__(self, wallet_address):
        self.wallet_address = wallet_address
    
    def pay(self, amount):
        return f"Paid ${amount} via Bitcoin ({self.wallet_address[:8]}...)"

class ShoppingCart:
    """쇼핑 카트 - 결제 전략을 사용"""
    
    def __init__(self):
        self.items = []
        self.payment_strategy = None
    
    def add_item(self, item, price):
        self.items.append((item, price))
        print(f"Added {item}: ${price}")
    
    def set_payment_strategy(self, strategy: PaymentStrategy):
        self.payment_strategy = strategy
    
    def checkout(self):
        total = sum(price for _, price in self.items)
        print(f"\nTotal: ${total}")
        
        if self.payment_strategy is None:
            print("Please select a payment method")
            return
        
        result = self.payment_strategy.pay(total)
        print(result)
        print("Purchase complete!\n")

# 사용 예시
cart = ShoppingCart()
cart.add_item("Laptop", 1200)
cart.add_item("Mouse", 25)
cart.add_item("Keyboard", 75)

# 신용카드로 결제
cart.set_payment_strategy(CreditCardPayment("1234-5678-9012-3456"))
cart.checkout()

# 다른 카트, 다른 결제 방식
cart2 = ShoppingCart()
cart2.add_item("Book", 30)
cart2.set_payment_strategy(PayPalPayment("user@example.com"))
cart2.checkout()

## 3.3 코드 품질

### Type Hints

인자, 반환값, 변수에 기대하는 타입을 적는다. IDE의 자동완성과 정적 타입 검사 도구가 이를 활용한다. 호출한 쪽에서 어떤 값을 주고받는지 파악하는 데도 도움이 된다.


In [ ]:
# Type Hints 기본
print("=== Type Hints ===")

from typing import List, Dict, Optional, Union, Tuple

def greet(name: str) -> str:
    """간단한 타입 힌트"""
    return f"Hello, {name}!"

def process_items(items: List[int]) -> Dict[str, int]:
    """리스트와 딕셔너리 타입 힌트"""
    return {
        'sum': sum(items),
        'count': len(items),
        'average': sum(items) // len(items) if items else 0
    }

def find_user(user_id: int) -> Optional[str]:
    """Optional: 값이 있을 수도, None일 수도 있음"""
    users = {1: 'Alice', 2: 'Bob', 3: 'Charlie'}
    return users.get(user_id)

def parse_value(value: Union[int, str]) -> int:
    """Union: 여러 타입 중 하나"""
    if isinstance(value, str):
        return int(value)
    return value

def get_coordinates() -> Tuple[float, float]:
    """Tuple: 고정된 길이의 튜플"""
    return (37.5665, 126.9780)

# 테스트
print(greet("Alice"))
print(process_items([1, 2, 3, 4, 5]))
print(find_user(1))
print(find_user(999))
print(parse_value("42"))
print(get_coordinates())

In [ ]:
# 클래스에 Type Hints 적용
print("\n=== 클래스 Type Hints ===")

from typing import List, Optional
from dataclasses import dataclass

@dataclass
class Student:
    name: str
    age: int
    grades: List[float]
    email: Optional[str] = None
    
    def average_grade(self) -> float:
        if not self.grades:
            return 0.0
        return sum(self.grades) / len(self.grades)
    
    def is_passing(self, threshold: float = 60.0) -> bool:
        return self.average_grade() >= threshold

# 사용
student = Student(
    name="Alice",
    age=20,
    grades=[85.5, 90.0, 88.5],
    email="alice@example.com"
)

print(f"Student: {student.name}")
print(f"Average: {student.average_grade():.2f}")
print(f"Passing: {student.is_passing()}")

### Docstrings

함수, 클래스, 모듈의 첫 문자열에 사용 방법을 적는다. 필요한 인자, 반환값, 발생할 수 있는 예외를 설명하고, 복잡한 경우에는 호출 예를 덧붙인다. Google, NumPy, Sphinx 등 프로젝트에서 정한 형식을 따른다.


In [ ]:
# Google Style Docstrings
print("=== Docstrings (Google Style) ===")

def calculate_statistics(numbers: List[float]) -> Dict[str, float]:
    """Calculate basic statistics for a list of numbers.
    
    This function computes the mean, median, and standard deviation
    of a given list of numbers.
    
    Args:
        numbers: A list of numbers to analyze. Must not be empty.
    
    Returns:
        A dictionary containing:
            - 'mean': The arithmetic mean
            - 'median': The median value
            - 'std': The standard deviation
    
    Raises:
        ValueError: If the list is empty.
        TypeError: If input is not a list of numbers.
    
    Examples:
        >>> calculate_statistics([1, 2, 3, 4, 5])
        {'mean': 3.0, 'median': 3, 'std': 1.4142}
        
        >>> calculate_statistics([10, 20, 30])
        {'mean': 20.0, 'median': 20, 'std': 8.1649}
    """
    if not numbers:
        raise ValueError("Cannot calculate statistics of empty list")
    
    n = len(numbers)
    mean = sum(numbers) / n
    
    sorted_numbers = sorted(numbers)
    if n % 2 == 0:
        median = (sorted_numbers[n//2-1] + sorted_numbers[n//2]) / 2
    else:
        median = sorted_numbers[n//2]
    
    variance = sum((x - mean) ** 2 for x in numbers) / n
    std = variance ** 0.5
    
    return {
        'mean': mean,
        'median': median,
        'std': std
    }

# Docstring 확인
print("Function docstring:")
print(calculate_statistics.__doc__)

# 사용
result = calculate_statistics([1, 2, 3, 4, 5])
print(f"\nResult: {result}")

## 종합 연습 문제

다음 프로그램을 작성하세요.

1. **로깅 시스템**:
   - Singleton Logger 구현
   - 로그 레벨별 decorator (@debug, @info, @error)
   - Type hints와 docstrings 추가

2. **알림 시스템**:
   - Observer 패턴 구현
   - 이메일, SMS, Push 알림 클래스
   - Factory 패턴으로 알림 생성

3. **데이터 처리 파이프라인**:
   - Strategy 패턴으로 처리 방식 선택
   - @lru_cache로 결과 캐싱
   - Context Manager로 리소스 관리


In [ ]:
# 종합 연습 문제 1: 로깅 시스템
print("=== 연습 문제 1: 로깅 시스템 ===")

# 여기에 구현하세요



In [ ]:
# 종합 연습 문제 2: 알림 시스템
print("\n=== 연습 문제 2: 알림 시스템 ===")

# 여기에 구현하세요



In [ ]:
# 종합 연습 문제 3: 데이터 처리 파이프라인
print("\n=== 연습 문제 3: 데이터 처리 파이프라인 ===")

# 여기에 구현하세요



## 복습

1. 같은 작업을 LBYL과 EAFP로 작성하고 정상 흐름과 오류 처리를 비교해 보자.
2. Decorator 적용 전후에 함수의 이름과 Docstring을 확인하고 `@wraps`의 역할을 설명해 보자.
3. 인자를 받는 Decorator에서 Factory, Decorator, Wrapper가 각각 언제 실행되는가?
4. Singleton, Factory, Observer, Strategy가 해결하려는 문제를 예제와 연결해 보자.
5. 예제 함수 하나에 Type Hint와 Docstring을 추가하고 사용 방법이 충분히 드러나는지 확인해 보자.


## 적용할 때 확인할 점

공통 동작을 여러 함수에 덧붙이면 Decorator, 객체 생성 방식을 묶으면 Factory, 변경을 구독자에게 알리면 Observer, 처리 방법을 교체하면 Strategy를 검토한다.

패턴을 적용한 뒤에는 호출 코드가 더 읽기 쉬워졌는지 확인한다. 작은 함수나 단순한 데이터 구조로 충분하다면 그 방식으로 작성해도 된다.


# 추가 학습 자료

- **필수:**
  - Regular Expression: https://ds31x.tistory.com/21
  - Regular Expression re: https://dsaint31.tistory.com/548
  - Python 정리: https://ds31x.tistory.com/page/Python-Python-%EC%A0%95%EB%A6%AC

- **권장:**
  - PEP 8: https://www.python.org/dev/peps/pep-0008/
  - Python Design Patterns: https://python-patterns.guide/
  - Type Hints: https://docs.python.org/3/library/typing.html
  - functools: https://docs.python.org/3/library/functools.html
